In [ ]:
# NOTE: Transformers is pre-installed in the Docker container via requirements.txt.
# No need to run %pip install here.

# Cohere Transcribe Evaluation

This notebook evaluates the Cohere Transcribe model (cohere-transcribe-03-2026) on a dataset of audio segments.
It uses the direct batch inference approach with the shared evaluation runner.

In [ ]:
# @title Install packages

import sys
import torch

# Monkeypatch torch to mock float8_e8m0fnu if missing (fixes transformers FP8 import crash on PyTorch 2.5.1)
if not hasattr(torch, "float8_e8m0fnu"):
    if hasattr(torch, "float8_e4m3fn"):
        torch.float8_e8m0fnu = torch.float8_e4m3fn
    else:
        torch.float8_e8m0fnu = torch.float16

import builtins

# The Magic Hack: Create a dummy class and inject it into Python's builtins
# so the Python 3.12 type-hint evaluator finds it and stops crashing.
class DummyPeftConfig:
    pass

builtins.PeftConfigLike = DummyPeftConfig

import os
import json
from transformers import pipeline
from google.cloud import storage
from huggingface_hub import login

# Import common utils
from common.gcs_utils import download_jsonl_manifest, upload_inference_results
from common.inference_pipeline_runner import run_inference_pipeline
from common.audio_utils import preprocess_audio_for_model

# Configure logging
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

In [ ]:
# --- Configuration ---
MODEL_NAME = "CohereLabs/cohere-transcribe-03-2026"
SELECTED_MODEL_KEY = "cohere_transcribe_03_2026"

# @markdown ### Google Cloud Platform Settings
GCP_PROJECT_ID = "<YOUR_GCP_PROJECT_ID>"  # @param {type:"string"}
GCS_MANIFEST_URI = "<YOUR_GCS_MANIFEST_URI>"  # @param {type:"string"}
GCS_BUCKET = "<YOUR_GCS_BUCKET_NAME>"  # @param {type:"string"}

# @markdown ### Evaluation Tracking
PROJECT_NAME = "<YOUR_PROJECT_NAME>"  # @param {type:"string"}
EXPERIMENT_NAME = "<YOUR_EXPERIMENT_NAME>"  # @param {type:"string"}

# @markdown ### Inference Parameters
BATCH_SIZE = 4  # @param {type:"integer"}
LIMIT = 10  # @param {type:"integer"}

# Log in to Hugging Face.
# This expects 'HF_TOKEN' to be set in Google Colab Secrets or as an environment variable.
# If neither is found, it will securely prompt you for the token.
from common.auth_utils import login_to_huggingface

login_to_huggingface()

In [ ]:
# @title Load the model and processor
import torch
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

print(f"Loading Cohere model and processor: {MODEL_NAME}")

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    trust_remote_code=False,
)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16,  # Keep bfloat16 to save memory
    trust_remote_code=False,
)

In [ ]:
# @title Run Evaluation
from common.inference_pipeline_runner import run_huggingface_inference_pipeline
from common.prompts import COMMON_SYSTEM_PROMPT, COMMON_USER_INSTRUCTION

storage_client = storage.Client(project=GCP_PROJECT_ID)
manifest_data = download_jsonl_manifest(storage_client, GCS_MANIFEST_URI)

# Construct the custom Cohere ChatML prompt format
cohere_chat_prompt = f"<|system|>\n{COMMON_SYSTEM_PROMPT}\n<|user|>\n<|audio|> {COMMON_USER_INSTRUCTION}\n<|assistant|>\n"

# Run the optimized, parallel Hugging Face evaluation pipeline
results_list = run_huggingface_inference_pipeline(
    model=model,
    processor=processor,
    manifest_data=manifest_data,
    storage_client=storage_client,
    project_name=PROJECT_NAME,
    selected_model=SELECTED_MODEL_KEY,
    batch_size=BATCH_SIZE,
    limit=LIMIT,
    text_prompt=cohere_chat_prompt,  # Pass custom template here!
    max_new_tokens=256,
    processor_kwargs={"language": "en"},
)

In [ ]:
# Upload results directly to GCS from memory
gcs_uri = upload_inference_results(
    storage_client,
    GCS_BUCKET,
    PROJECT_NAME,
    SELECTED_MODEL_KEY,
    EXPERIMENT_NAME,
    results_list,
)